# PySpark + HDFS Demo

This notebook demonstrates reading data from HDFS using PySpark.

## 1. Setup: Create Sample Data and Upload to HDFS

First, let's create some sample data and upload it to HDFS.

In [1]:
import subprocess
import os

# Create sample CSV data
sample_data = """id,name,age,city
1,Alice,28,New York
2,Bob,35,San Francisco
3,Charlie,42,Chicago
4,Diana,31,Boston
5,Eve,26,Seattle
"""

# Save locally first
with open('/tmp/sample_data.csv', 'w') as f:
    f.write(sample_data)

print("Sample data created locally at /tmp/sample_data.csv")

Sample data created locally at /tmp/sample_data.csv


In [2]:
# Create HDFS directory and upload the file
# Note: Make sure HDFS is running (start-dfs.sh)

!hdfs dfs -mkdir -p /user/microlab
!hdfs dfs -put -f /tmp/sample_data.csv /user/microlab/
!hdfs dfs -ls /user/microlab/

fish: Unknown command: hdfs
fish: 
hdfs dfs -mkdir -p /user/microlab
^~~^
fish: Unknown command: hdfs
fish: 
hdfs dfs -put -f /tmp/sample_data.csv /user/microlab/
^~~^
fish: Unknown command: hdfs
fish: 
hdfs dfs -ls /user/microlab/
^~~^


## 2. Initialize PySpark Session

In [1]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("HDFS Demo") \
    .master("local[*]") \
    .config("spark.hadoop.sun.security.auth.login.config", "true") \
    .config("spark.driver.extraJavaOptions", "--add-opens java.base/javax.security.auth=ALL-UNNAMED") \
    .getOrCreate()


print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 16:23:44 WARN Utils: Your hostname, shamil.local, resolves to a loopback address: 127.0.0.1; using 172.28.124.183 instead (on interface feth2378)
26/03/07 16:23:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

26/03/07 16:23:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2446)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2446)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:339)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:483)
	at py4j.reflection.MethodInvoker.in

## 3. Read Data from HDFS

In [ ]:
# Read CSV from HDFS
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/user/microlab/sample_data.csv")

print("Data successfully read from HDFS!")
print(f"Number of rows: {df.count()}")

In [ ]:
# Display the schema
df.printSchema()

In [ ]:
# Show the data
df.show()

## 4. Basic PySpark Operations

In [ ]:
# Filter: people older than 30
df.filter(df.age > 30).show()

In [ ]:
# Group by and aggregate
from pyspark.sql.functions import avg, count

print(f"Average age: {df.agg(avg('age')).collect()[0][0]:.1f}")
print(f"Total records: {df.count()}")

In [ ]:
# SQL query on the data
df.createOrReplaceTempView("people")

spark.sql("""
    SELECT city, COUNT(*) as count, AVG(age) as avg_age
    FROM people
    GROUP BY city
    ORDER BY avg_age DESC
""").show()

## 5. Cleanup

In [ ]:
# Stop the Spark session
spark.stop()
print("Spark session stopped.")